#1. Carga y preparación de datos (ETL)

In [ ]:
# 1.1 Librerías
import re
import sys
from pathlib import Path

_nb_dir = Path(".").resolve()
_project_root = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from config import PROCESSED_DATASET_PATH, PROCESSED_DIR

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import gaussian_kde

# Constante: nombre de columna de lectura del sensor (dominio)
COL_READING = "Leyendo"

# Patrón archivos por mes-año: nombre base + _YYYY-MM.csv
_processed_stem = PROCESSED_DATASET_PATH.stem  # ej. Procesamiento_de_Datos
_monthly_pattern = re.compile(
    "^" + re.escape(_processed_stem) + r"_(\d{4})-(\d{2})\.csv$"
)


def _load_and_clean(path):
    """Carga un CSV/Excel y aplica limpieza común (Leyendo, fechas)."""
    if str(path).endswith(".csv"):
        out = pd.read_csv(path)
    else:
        out = pd.read_excel(path)
    out[COL_READING] = pd.to_numeric(
        out[COL_READING].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    out["Tiempo de Inicio"] = pd.to_datetime(out["Tiempo de Inicio"])
    return out


# 1.2 Detectar archivos por mes-año en data/processed
monthly_dfs = []
use_fallback = False
monthly_files = []

for f in Path(PROCESSED_DIR).iterdir():
    if not f.is_file():
        continue
    m = _monthly_pattern.match(f.name)
    if m:
        monthly_files.append((int(m.group(1)), int(m.group(2)), f))

monthly_files.sort(key=lambda x: (x[0], x[1]))

for year, month, path in monthly_files:
    try:
        part = _load_and_clean(path)
        if part.empty:
            use_fallback = True
            break
        monthly_dfs.append(part)
    except Exception:
        use_fallback = True
        break

# Si no hay archivos mes-año o hubo error: usar consolidado como fallback
if not monthly_files or use_fallback:
    if str(PROCESSED_DATASET_PATH).endswith(".csv"):
        df = pd.read_csv(PROCESSED_DATASET_PATH)
    else:
        df = pd.read_excel(PROCESSED_DATASET_PATH)
    df[COL_READING] = pd.to_numeric(
        df[COL_READING].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df["Tiempo de Inicio"] = pd.to_datetime(df["Tiempo de Inicio"])
    if use_fallback and monthly_files:
        _source = "consolidado (fallback por error en archivos mes-año)"
    else:
        _source = "consolidado (sin archivos mes-año)"
else:
    df = pd.concat(monthly_dfs, ignore_index=True)
    _source = f"{len(monthly_dfs)} archivo(s) mes-año"

# 1.3 Ordenar cronológicamente
df = df.sort_values("Tiempo de Inicio")

# Resumen de carga
print(f"Origen: {_source}. Filas: {len(df)}")
df.head()

Origen: 9 archivo(s) mes-año. Filas: 13894


,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura
0,CO,Sensor SN: 2206309128#S0001,Periodo 1,Monóxido de carbono,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:38:29,NaN,NaN,NaN,NaN,20.0
1,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0
2,O2,Sensor SN: 21071TQ142#S0020,Periodo 1,Oxígeno,20.40,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0
3,LEL(CH4),Sensor SN: 1908138189#S0021,Periodo 1,Gas combustible,0.05,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0
4,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,-0.10,ppm,NaN,NaN,2023-03-31 14:37:19,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0


#2. EDA – Análisis estadístico descriptivo

In [30]:
df["Leyendo"].describe()

count    13894.000000
mean         5.658797
std         21.440899
min        -21.000000
25%          0.000000
50%          0.210000
75%          4.000000
max        350.000000
Name: Leyendo, dtype: float64

2.1 Estadísticos generales

In [31]:
df["Leyendo"].describe()

count    13894.000000
mean         5.658797
std         21.440899
min        -21.000000
25%          0.000000
50%          0.210000
75%          4.000000
max        350.000000
Name: Leyendo, dtype: float64

2.2 Estadísticos por gas

In [32]:
df.groupby("Gas")["Leyendo"].describe()

,count,mean,std,min,25%,50%,75%,max
Gas,,,,,,,,
Dióxido de carbono,4698.0,0.489977,0.908723,-0.17,0.02,0.09,0.627500,5.0
Dióxido de nitrógeno,748.0,0.467513,1.723505,-0.90,-0.40,0.00,0.300000,8.6
Gas combustible,491.0,0.551752,4.966481,-0.14,0.00,0.11,0.130000,61.0
Monóxido de carbono,2716.0,7.504786,45.315464,-21.00,-5.00,0.00,3.000000,350.0
Oxígeno,2694.0,19.759465,2.222973,0.00,19.60,20.40,20.700001,30.0
Sulfuro de hidrógeno,2547.0,0.818924,2.585439,-1.60,0.00,0.20,0.600000,24.1


2.3 Conteo de registros por gas

In [33]:
df["Gas"].value_counts()

Gas
Dióxido de carbono      4698
Monóxido de carbono     2716
Oxígeno                 2694
Sulfuro de hidrógeno    2547
Dióxido de nitrógeno     748
Gas combustible          491
Name: count, dtype: int64

#3️. EDA – Análisis de distribuciones

3.1 Histogramas por gas

In [34]:
gases = df["Gas"].unique()

for gas in gases:
    subset = df[df["Gas"] == gas]["Leyendo"].dropna()

    if len(subset) > 1:
        kde = gaussian_kde(subset)
        x_range = np.linspace(subset.min(), subset.max(), 200)

        fig = go.Figure()

        fig.add_trace(go.Histogram(
            x=subset,
            nbinsx=30,
            histnorm="density",
            name="Frecuencia",
            marker_color="steelblue",
            opacity=0.6
        ))

        fig.add_trace(go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode="lines",
            name="KDE",
            line=dict(color="crimson", width=2)
        ))

        fig.update_layout(
            title=f"Distribución de concentración — {gas}",
            xaxis_title="Concentración",
            yaxis_title="Densidad",
            template="plotly_white",
            hovermode="x unified"
        )

        fig.show()

3.2 Boxplot interactivo

In [35]:
fig_box = px.box(
    df,
    x="Gas",
    y="Leyendo",
    color="Gas",
    template="plotly_white",
    title="Detección de Outliers por Gas"
)

fig_box.update_layout(showlegend=False)
fig_box.show()

#4. Análisis Temporal

4.1 Serie temporal por gas

In [36]:
for gas in gases:
    subset = df[df["Gas"] == gas]

    fig = px.line(
        subset,
        x="Tiempo de Inicio",
        y="Leyendo",
        title=f"Serie Temporal — {gas}",
        template="plotly_white"
    )

    fig.show()

4.2 Tendencia con media móvil

In [37]:
df["Media_Movil"] = df["Leyendo"].rolling(window=20).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Leyendo"],
    mode="lines",
    name="Valor original",
    line=dict(color="lightgray")
))

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Media_Movil"],
    mode="lines",
    name="Media móvil",
    line=dict(color="red", width=2)
))

fig.update_layout(
    title="Tendencia con Media Móvil",
    template="plotly_white"
)

fig.show()

4.3 Detección de cambios abruptos

In [38]:
df["Cambio"] = df["Leyendo"].diff().abs()

threshold = df["Cambio"].mean() + 2 * df["Cambio"].std()
df["Cambio_Brusco"] = df["Cambio"] > threshold

df[df["Cambio_Brusco"] == True].head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
4374,CO,Sensor SN: 160541U005#S0001,Periodo 1,Monóxido de carbono,73.00,ppm,"5,70","0,52",2024-11-20 05:38:19,2024-11-20 05:38:29,NaN,NaN,NaN,Alta,25.0,16.3075,71.80,True
4375,CO2,Sensor SN: 17020PE013#S0011,Periodo 1,Dióxido de carbono,0.24,%vol,"0,10","0,02",2024-11-20 05:38:19,2024-11-20 05:38:49,NaN,NaN,NaN,NaN,25.0,15.1695,72.76,True
4377,CO,Sensor SN: 160541U005#S0001,Periodo 1,Monóxido de carbono,85.00,ppm,"6,62","0,55",2024-11-20 05:38:29,2024-11-20 05:38:39,NaN,NaN,NaN,Alta,25.0,19.4095,84.76,True
4378,H2S,Sensor SN: 160541U005#S0002,Periodo 1,Sulfuro de hidrógeno,4.60,ppm,"0,23","0,09",2024-11-20 05:38:29,2024-11-20 05:38:39,NaN,NaN,NaN,Baja,25.0,19.5595,80.40,True
4379,CO,Sensor SN: 160541U005#S0001,Periodo 1,Monóxido de carbono,93.00,ppm,"7,63","0,58",2024-11-20 05:38:39,2024-11-20 05:38:49,NaN,NaN,NaN,Alta,25.0,22.8495,91.70,True


#5. EDA – Análisis multivariable

5.1 Pivotear datos por gas

In [39]:
df_pivot = df.pivot_table(
    index="Tiempo de Inicio",
    columns="Gas",
    values="Leyendo"
)

5.2 Matriz de correlación interactiva

In [40]:
fig_corr = px.imshow(
    df_pivot.corr(),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Matriz de Correlación entre Gases"
)

fig_corr.show()

#6. EDA – Patrones previos a eventos

6.1 Definir evento (Percentil 95)

In [41]:
limite = df["Leyendo"].quantile(0.95)
eventos = df[df["Leyendo"] > limite]

eventos.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
39,O2,Sensor SN: 21071TQ142#S0020,Periodo 3,Oxígeno,20.900000,%vol,NaN,NaN,2023-03-31 14:42:17,2023-03-31 14:42:27,NaN,NaN,NaN,NaN,20.0,8.1790,0.199999,False
51,O2,Sensor SN: 22101YV009#S0020,Periodo 1,Oxígeno,20.900000,%vol,NaN,NaN,2024-09-06 12:43:49,2024-09-06 12:50:29,NaN,NaN,NaN,NaN,31.0,5.1955,20.900000,False
52,O2,Sensor SN: 22101YV009#S0020,Periodo 1,Oxígeno,21.000000,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29.0,5.2020,21.000000,False
59,O2,Sensor SN: 22101YV009#S0020,Periodo 2,Oxígeno,21.900000,%vol,NaN,NaN,2024-09-06 14:40:40,2024-09-06 14:41:20,NaN,NaN,NaN,NaN,26.0,4.2220,21.800000,False
65,O2,Sensor SN: 22101YV009#S0020,Periodo 2,Oxígeno,21.799999,%vol,NaN,NaN,2024-09-06 14:41:20,2024-09-06 14:41:30,NaN,NaN,NaN,NaN,26.0,4.3000,21.499999,False


6.2 Analizar ventana previa de 30 minutos

In [42]:
ventanas = []

for tiempo in eventos["Tiempo de Inicio"]:
    ventana = df[
        (df["Tiempo de Inicio"] >= tiempo - pd.Timedelta(minutes=30)) &
        (df["Tiempo de Inicio"] <= tiempo)
    ]
    ventanas.append(ventana)

df_ventanas = pd.concat(ventanas)

df_ventanas.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
0,CO,Sensor SN: 2206309128#S0001,Periodo 1,Monóxido de carbono,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:38:29,NaN,NaN,NaN,NaN,20.0,NaN,NaN,False
1,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0,NaN,0.00,False
2,O2,Sensor SN: 21071TQ142#S0020,Periodo 1,Oxígeno,20.40,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0,NaN,20.40,False
3,LEL(CH4),Sensor SN: 1908138189#S0021,Periodo 1,Gas combustible,0.05,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0,NaN,20.35,False
4,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,-0.10,ppm,NaN,NaN,2023-03-31 14:37:19,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0,NaN,0.15,False
